In [ ]:
import os
import pandas as pd
import hashlib
from scapy.all import rdpcap, IP, TCP, UDP, Ether, ARP, ICMP

# Set a sample file for prototyping
sample_pcap = '../data/raw/captures_IoT-Sentinel/Aria/Setup-A-1-STA.pcap'
print(f"Reading {sample_pcap}...")
packets = rdpcap(sample_pcap)
print(f"Loaded {len(packets)} packets.")

In [ ]:
FLOW_TIMEOUT = 120 # Timeout in seconds

flows = {}
packet_data = []

for pkt in packets:
    pkt_info = {
        'time': float(pkt.time),
        'length': len(pkt),
        'src_mac': None,
        'dst_mac': None,
        'src_ip': None,
        'dst_ip': None,
        'src_port': None,
        'dst_port': None,
        'protocol': None,
        'tcp_flags': None,
        'flow_id': None
    }
    
    # Ethernet layer
    if Ether in pkt:
        pkt_info['src_mac'] = pkt[Ether].src
        pkt_info['dst_mac'] = pkt[Ether].dst
        
    # IP Layer
    if IP in pkt:
        pkt_info['src_ip'] = pkt[IP].src
        pkt_info['dst_ip'] = pkt[IP].dst
        pkt_info['protocol'] = pkt[IP].proto
        
        # TCP/UDP
        if TCP in pkt:
            pkt_info['src_port'] = pkt[TCP].sport
            pkt_info['dst_port'] = pkt[TCP].dport
            pkt_info['tcp_flags'] = str(pkt[TCP].flags)
        elif UDP in pkt:
            pkt_info['src_port'] = pkt[UDP].sport
            pkt_info['dst_port'] = pkt[UDP].dport
            
    # Determine Flow ID based on 5-tuple + Timeout
    if pkt_info['src_ip'] and pkt_info['dst_ip']:
        # Sort endpoints to ensure bidirectional traffic goes into the same flow
        p1 = f"{pkt_info['src_ip']}:{pkt_info['src_port']}"
        p2 = f"{pkt_info['dst_ip']}:{pkt_info['dst_port']}"
        endpoints = sorted([p1, p2])
        flow_key = f"{endpoints[0]}-{endpoints[1]}-{pkt_info['protocol']}"
        
        # Check if flow exists and if timeout is exceeded
        if flow_key in flows:
            last_time, flow_hash = flows[flow_key]
            if pkt_info['time'] - last_time > FLOW_TIMEOUT:
                # New flow started
                flow_hash = hashlib.md5(f"{flow_key}-{pkt_info['time']}".encode()).hexdigest()
        else:
            # Create new flow hash
            flow_hash = hashlib.md5(f"{flow_key}-{pkt_info['time']}".encode()).hexdigest()
            
        # Update flow last seen time
        flows[flow_key] = (pkt_info['time'], flow_hash)
        pkt_info['flow_id'] = flow_hash
        
    packet_data.append(pkt_info)

df = pd.DataFrame(packet_data)
df.head(15)